# **Estimasi Cepat Sebaran dan Dampak Banjir (version 2)**

Berbasis integrasi antara *Height Above Nearest Drainage* (HAND) dan data koordinat kedalaman genangan banjir lapangan.

Output:
1.   Raster kedalaman genangan banjir
2.   Poligon sebaran genangan banjir
3.   Data luas dan persentase genangan banjir per wilayah administrasi
4.   Estimasi penduduk terdampak banjir
5.   Estimasi bangunan terdampak banjir
6.   Estimasi penutupan lahan terdampak banjir



Created by: Syam'ani (https://github.com/syamaniulm) &copy; 2026

License: GNU General Public License v3.0 (https://www.gnu.org/licenses/gpl-3.0.html)

**Sitasi:**<br>

Penggunaan HAND di dalam dokumen resmi wajib mengutip literatur ini: https://www.sciencedirect.com/science/article/abs/pii/S0022169411002599<br>
Penggunaan FABDEM di dalam dokumen resmi wajib mengutip literatur ini: https://iopscience.iop.org/article/10.1088/1748-9326/ac4d4f

# **Peringatan!**

Kode program ini merupakan proyek eksperimental. Sehingga masih memerlukan validasi di lapangan.

In [ ]:
!pip install -U geemap

## **Tahap 1: Persiapan**

In [ ]:
import ee, geemap, pandas as pd, geopandas as gpd

In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(project='ee-syamaniulm')      # Sesuaikan 'your-ee-project' dengan nama Google Earth Engine Project Anda

In [ ]:
# Pengaturan nama folder data di dalam Google Drive

folder = 'flood'    # Tentukan nama folder data Anda sesuai dengan nama folder di dalam Google Drive

In [ ]:
# Penentuan datum geodetik dan zona UTM, untuk keperluan kalkulasi luas wilayah
# Kode EPSG di bawah adalah untuk datum geodetik WGS 1984 dan UTM Zone 50S
# Tentukan datum geodetik yang diinginkan dan sesuaikan zona UTM dengan wilayah kerja Anda, buka laman https://epsg.io/

utm_epsg = '32750'

In [ ]:
# Mengakses Google Drive dan penentuan path

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

path = '/content/drive/MyDrive/' + folder + '/'

In [ ]:
# Membuka file data koordinat kedalaman genangan banjir lapangan

flood_locations = pd.read_csv(path + 'Flood_Locations.csv')      # Sesuaikan nama filenya
flood_locations_gdf = gpd.GeoDataFrame(flood_locations, geometry=gpd.points_from_xy(flood_locations.Long, flood_locations.Lat), crs='EPSG:4326')
flood_locations_gdf.head()

## **Tahap 2: Estimasi Sebaran Genangan Banjir**

In [ ]:
# Konversi tabel data banjir lapangan menjadi fitur titik

flood_points = geemap.geopandas_to_ee(flood_locations_gdf)
flood_points

In [ ]:
# Mengakses data catchment area dan HAND
# Catatan: Nilai HAND dalam desimeter

c_area = '50k'        # Pilih di antara '5k', '10k', '25k', '50k', '100k' atau '250k'

ca = ee.FeatureCollection('projects/ee-syamaniulm/assets/carea').filterBounds(flood_points).filter(ee.Filter.eq('id', 'ca' + c_area))
hand = ee.Image('projects/ee-syamaniulm/assets/hand').select('h' + c_area).rename('HAND')

In [ ]:
# Mengambil nilai HAND untuk setiap titik banjir lapangan

flood_location_hand_values = hand.sampleRegions(collection=flood_points, scale=hand.projection().nominalScale()).aggregate_array('HAND').getInfo()
flood_location_hand_values  = [hand_values/10 if hand_values <= 250 else 0 for hand_values in flood_location_hand_values]

In [ ]:
# Menghitung total kedalaman genangan dari permukaan sungai terdekat
# Total kedalaman genangan = kedalaman banjir lapangan + HAND

flood_locations['Hand'] = flood_location_hand_values
flood_locations['Total_Depth'] = flood_locations['Depth'] + flood_locations['Hand']
flood_locations.head()

In [ ]:
# Analisis statistik untuk data total kedalaman genangan yang tidak wajar (anomali)
# menggunakan metode Interquartile Range (IQR)

import numpy as np

total_depth = np.array(flood_locations['Total_Depth'].to_list())

q1 = np.percentile(total_depth, 25)
q3 = np.percentile(total_depth, 75)

iqr = q3 - q1

upper_bound = (q3 + 1.5 * iqr).item()
upper_bound

In [ ]:
# Jika diperlukan, buang anomali data total kedalaman genangan

flood_location_filtered = flood_locations.drop(flood_locations[flood_locations['Hand'] == 0].index)
flood_location_filtered = flood_location_filtered.drop(flood_location_filtered[flood_location_filtered['Total_Depth'] > upper_bound].index)   # Komentari baris ini jika semua data dipakai (tidak ada anomali yang dibuang)
flood_location_filtered.head()

In [ ]:
# Konversi data total kedalaman genangan menjadi fitur titik

flood_location_filtered_gdf = gpd.GeoDataFrame(flood_location_filtered, geometry=gpd.points_from_xy(flood_location_filtered.Long, flood_location_filtered.Lat), crs='EPSG:4326')
flood_location_filtered_feature = geemap.geopandas_to_ee(flood_location_filtered_gdf)
flood_location_filtered_feature

In [ ]:
# Interseksi antara catchment area dan fitur titik total kedalaman genangan
# untuk mendapatkan total kedalaman genangan maksimum per catchment area

def max_depth_per_ca(feature):
    ca_geom = feature.geometry()
    intersected = flood_location_filtered_feature.filterBounds(ca_geom)
    max_depth = ee.List(intersected.aggregate_array('Total_Depth')).reduce(ee.Reducer.max())
    return feature.set('Max_Total_Depth', max_depth)

ca_with_max_depth = ca.map(max_depth_per_ca)
ca_with_max_depth

In [ ]:
# Kalibrasi nilai HAND dari satuan desimeter ke meter

hand_flood = hand.updateMask(hand.lte(250)).divide(10)

In [ ]:
# Konversi catchment area menjadi total kedalaman genangan maksimum

hand_ref = hand_flood.select('HAND').projection()

max_depth_image = (
    ca_with_max_depth
    .reduceToImage(properties=['Max_Total_Depth'], reducer=ee.Reducer.first())
    .rename('Max_Total_Depth')
    .reproject(crs=hand_ref.crs(), scale=hand_ref.nominalScale())
)

max_depth_image

In [ ]:
# Ekstraksi data kedalaman genangan banjir
# Kedalaman genangan = Total kedalaman genangan maksimum - HAND

flood_depth = max_depth_image.subtract(hand_flood).rename('Flood_Depth')
flood_depth = flood_depth.updateMask(flood_depth.gt(0)).updateMask(hand_flood.gt(0))

In [ ]:
# Ekstraksi poligon sebaran genangan banjir
# Peringatan: Proses ini dapat memakan waktu yang sangat lama, bahkan gagal, jika area tergenang banjir sangat luas

flooded_area = flood_depth.gt(0).selfMask().reduceToVectors(
    scale=flood_depth.projection().nominalScale(),
    geometry=flood_depth.geometry(),
    geometryType='polygon',
    eightConnected=True,
    labelProperty='mask',
    maxPixels=1e13
)

flooded_area

In [ ]:
# Jika diperlukan, data kedalaman genangan banjir dapat diekspor ke GeoTIFF
# File akan tersimpan di dalam Google Drive
# Peringatan: Ekspor bisa gagal jika wilayah genangan banjir cukup luas (ukuran file terlalu besar)
# Kunjungi laman https://geemap.org/notebooks/118_download_image/ untuk referensi ekspor file raster yang cukup besar

output_geotiff_path = path + 'flood_depth.tif'  # Sesuaikan nama filenya
geemap.ee_export_image(flood_depth, filename=output_geotiff_path, scale=flood_depth.projection().nominalScale(), region=flooded_area.geometry(), file_per_band=False)

In [ ]:
# Jika diperlukan, poligon sebaran genangan banjir dapat diekspor menjadi shapefile
# Shapefile akan tersimpan ke dalam Google Drive

flooded_area_gdf = geemap.ee_to_gdf(flooded_area).dissolve(by='mask')
output_shp_path = path + 'flooded_area.shp'      # Sesuaikan nama filenya
flooded_area_gdf.to_file(output_shp_path, driver='ESRI Shapefile')

In [ ]:
# Ekstraksi data kedalaman banjir maksimum hasil estimasi

import math

max_flood_depth = math.floor(geemap.image_stats(flood_depth, region=ca.geometry(), scale=flood_depth.projection().nominalScale()).getInfo()['max']['Flood_Depth'])
max_flood_depth

In [ ]:
# Mengakses fitur data administrasi
# Catatan: Fitur data administrasi bersumber dari Badan Informasi Geospasial (BIG) (https://tanahair.indonesia.go.id/portal-web/)

admin = ee.FeatureCollection('projects/ee-syamaniulm/assets/admin').filterBounds(flooded_area)

## **Tahap 3: Visualisasi Sebaran Genangan Banjir**

In [ ]:
# Visualisasi sebaran dan kedalaman genangan banjir hasil estimasi

flood_vis = {'min': 0, 'max': max_flood_depth, 'palette': 'Blues'}
flood_style = {'color': 'blue', 'fillColor': '00000000', 'width': 1}
admin_style = {'color': 'red', 'fillColor': '00000000', 'width': 1}

Map = geemap.Map(basemap='HYBRID')
Map.centerObject(flooded_area, 12)
Map.addLayer(flood_depth, flood_vis, 'Flood Depth', opacity=0.7)
Map.addLayer(flood_location_filtered_feature, {'color': 'red'}, 'Flood Locations', shown=False)
Map.addLayer(flooded_area.style(**flood_style), {}, 'Flooded Area', shown=False)
Map.addLayer(admin.style(**admin_style), {}, 'Administrative Boundaries', shown=False)
# Map.add_labels(data=admin, column='NAMOBJ', font_size='10pt', font_color='yellow', layer_name='Desa/Kelurahan')
Map.add_colorbar(flood_vis,label='Flood Depth (m)', layer_name='Flood Depth', orientation='horizontal')
Map

In [ ]:
# Konversi fitur data administrasi menjadi GeoDataFrame dan kalkulasi luas wilayah administrasi

admin_gdf = geemap.ee_to_gdf(admin)
admin_gdf_proj = admin_gdf.to_crs('EPSG:' + utm_epsg)
admin_gdf_proj['AREA_HA'] = round(admin_gdf_proj.geometry.area / 10000, 2)
admin_gdf_proj = admin_gdf_proj[['WADMPR','WADMKK','WADMKC','WADMKD', 'AREA_HA', 'geometry']]

In [ ]:
# Tumpang susun (intersect) antara poligon administrasi dan poligon sebaran genangan banjir

flooded_area_gdf_proj = flooded_area_gdf.to_crs('EPSG:' + utm_epsg)
admin_flooded_area_gdf_proj = gpd.overlay(admin_gdf_proj, flooded_area_gdf_proj, how='intersection')

In [ ]:
# Kalkulasi luas wilayah tergenang banjir untuk setiap wilayah administrasi

admin_flooded_area_gdf_proj['FLOOD_AREA_HA'] = round(admin_flooded_area_gdf_proj.geometry.area / 10000, 2)
admin_flooded_area_gdf_proj['PERCENTAGE'] = round((admin_flooded_area_gdf_proj['FLOOD_AREA_HA'] / admin_flooded_area_gdf_proj['AREA_HA'])*100,2)
admin_flooded_area_gdf_proj = admin_flooded_area_gdf_proj[['WADMPR','WADMKK','WADMKC','WADMKD', 'AREA_HA', 'FLOOD_AREA_HA', 'PERCENTAGE']].rename(columns={'WADMPR': 'Provinsi', 'WADMKK': 'Kabupaten/Kota', 'WADMKC': 'Kecamatan', 'WADMKD': 'Desa/Kelurahan', 'AREA_HA': 'Luas Wilayah (hektare)', 'FLOOD_AREA_HA': 'Luas Genangan (hektare)', 'PERCENTAGE': 'Persentase Tergenang (%)'})

In [ ]:
# Ekstraksi dan visualisasi luas genangan banjir per desa/kelurahan

pd.set_option('display.max_rows', None)
admin_flooded_area_gdf_proj

In [ ]:
# Kalkulasi total desa/kelurahan terdampak banjir

total_impacted_admin = admin_gdf['KDEPUM'].nunique()
print(f"Total Jumlah Desa/Kelurahan Terdampak: {total_impacted_admin}")

In [ ]:
# Jika diperlukan, tabel luas area tergenang banjir per desa/kelurahan dapat diekspor ke file Excel
# File Excel akan tersimpan di dalam Google Drive

output_excel_path = path + 'estimated_flooded_admin_summary.xlsx'  # Sesuaikan nama filenya
admin_flooded_area_gdf_proj.to_excel(output_excel_path, index=False)

In [ ]:
# Ekstraksi dan visualisasi luas genangan banjir per kecamatan

kecamatan_summary = admin_flooded_area_gdf_proj.groupby('Kecamatan')['Luas Genangan (hektare)'].sum().reset_index()
kecamatan_summary

In [ ]:
# Ekstraksi dan visualisasi luas genangan banjir per kabupaten/kota

kabupaten_summary = admin_flooded_area_gdf_proj.groupby('Kabupaten/Kota')['Luas Genangan (hektare)'].sum().reset_index()
kabupaten_summary

## **Tahap 4: Estimasi Dampak Banjir**

### **Tahap 4a: Estimasi Dampak Banjir Terhadap Penduduk**

***Catatan:***

*Estimasi dampak banjir terhadap penduduk akan lebih akurat jika Anda memiliki data geospasial penduduk terbaru dan resolusi spasialnya lebih tinggi. Di sini data penduduk yang dijadikan sumber adalah WorldPop Global Population Data 2015-2030.*

***Sitasi:***

*Bondarenko M., Priyatikanto R., Tejedor-Garavito N., Zhang W., McKeen T., Cunningham A., Woods T., Hilton J., Cihan D., Nosatiuk B., Brinkhoff T., Tatem A., Sorichetta A.. 2025 Constrained estimates of 2015- 2030 total number of people per grid square at a resolution of 3 arc (approximately 100m at the equator) R2024B version v1. Global Demographic Data Project - Funded by The Bill and Melinda Gates Foundation (INV-045237). WorldPop - School of Geography and Environmental Science, University of Southampton. DOI:10.5258/SOTON/WP00803*

In [ ]:
# Akses data WorldPop Global Population Data 2015-2030 untuk wilayah Indonesia

year_pop = '2021'     # Sesuaikan waktunya dengan waktu kejadian banjir

population = (
    ee.ImageCollection('projects/sat-io/open-datasets/WORLDPOP/pop')
    .filter(ee.Filter.eq('year', year_pop))
    .filter(ee.Filter.eq('country_code', 'IDN'))
    .filterBounds(flooded_area)
).first().clip(flooded_area)

population

In [ ]:
# Kalkulasi populasi maksimum untuk keperluan visualisasi data geospasial WorldPop Global Population Data 2015-2030

max_pop = math.floor(geemap.image_stats(population, region=ca, scale=population.projection().nominalScale()).getInfo()['max']['population'])
max_pop

In [ ]:
# Visualisasi populasi penduduk terdampak banjir

pop_vis = {'min': 0, 'max': max_pop, 'palette': 'plasma'}

Map = geemap.Map()
Map.centerObject(flooded_area, 12)
Map.addLayer(population, pop_vis, 'Population')
Map.addLayer(flooded_area, {'color': '0000ff' + '50'}, 'Flooded Area', shown=False)
Map.add_colorbar(pop_vis, label='Affected Population Per Hectare', layer_name='Population', orientation='horizontal')
Map

In [ ]:
# Kalkulasi jumlah penduduk terdampak banjir per desa/kelurahan

pop_flooded = population.reduceRegions(
    collection=admin,
    reducer=ee.Reducer.sum(),
    scale=population.projection().nominalScale()
)

pop_flooded_gdf = geemap.ee_to_gdf(pop_flooded)
pop_flooded_gdf['sum'] = pop_flooded_gdf['sum'].fillna(0).astype(int)

In [ ]:
# Visualisasi tabel estimasi jumlah penduduk terdampak banjir per desa/kelurahan

est_pop_flooded_gdf  = pop_flooded_gdf [['WADMPR','WADMKK','WADMKC','WADMKD', 'sum']].rename(columns={'WADMPR': 'Provinsi', 'WADMKK': 'Kabupaten/Kota', 'WADMKC': 'Kecamatan', 'WADMKD': 'Desa/Kelurahan', 'sum': 'Jumlah Penduduk Terdampak'})
est_pop_flooded_gdf

In [ ]:
# Jika diperlukan, tabel estimasi jumlah penduduk terdampak banjir per desa/kelurahan dapat diekspor ke file Excel
# File Excel akan tersimpan di dalam Google Drive

output_excel_path = path + 'estimated_flooded_population_summary.xlsx'  # Sesuaikan nama filenya
est_pop_flooded_gdf.to_excel(output_excel_path, index=False)

In [ ]:
# Kalkulasi total penduduk terdampak banjir untuk seluruh wilayah administrasi

total_impacted_population = est_pop_flooded_gdf['Jumlah Penduduk Terdampak'].sum()
print(f"Estimasi Total Jumlah Penduduk Terdampak: {total_impacted_population}")

### **Tahap 4b: Estimasi Dampak Banjir Terhadap Bangunan**

***Catatan:***

*Estimasi dampak banjir terhadap bangunan akan lebih akurat jika Anda memiliki shapefile lokasi bangunan terbaru. Di sini data bangunan yang dijadikan sumber adalah Global Google-Microsoft Open Buildings Dataset.*


***Sitasi:***

*Google-Microsoft Open Buildings - combined by VIDA, https://beta.source.coop/repositories/vida/google-microsoft-open-buildings. Date Accessed: 2026-05-23*

In [ ]:
# Akses data Global Google-Microsoft Open Buildings Dataset untuk wilayah Indonesia

open_buildings = ee.FeatureCollection('projects/sat-io/open-datasets/VIDA_COMBINED/IDN')
flooded_buildings = open_buildings.filterBounds(flooded_area)

In [ ]:
# Visualisasi bangunan terdampak banjir

Map = geemap.Map()
Map.centerObject(flooded_area, 12)
Map.addLayer(flooded_buildings, {'color': 'purple'}, 'Buildings')
Map.addLayer(flooded_area, {'color': '0000ff' + '50'}, 'Flooded Area')
Map

In [ ]:
# Kalkulasi jumlah bangunan terdampak banjir per desa/kelurahan
# Lama proses kalkulasi tergantung jumlah desa/kelurahan terdampak banjir

from tqdm.auto import tqdm

admin_code_list = admin_gdf['KDEPUM'].to_list()
num_buildings = []

print('Proses kalkulasi dilakukan per desa/kelurahan yang terdampak banjir...')

for code in tqdm(admin_code_list):
  region = geemap.geopandas_to_ee(admin_gdf[admin_gdf['KDEPUM'] == code]).geometry()
  flooded_buildings_in_region = flooded_buildings.filterBounds(region)
  num_buildings.append(flooded_buildings_in_region.size().getInfo())

In [ ]:
# Visualisasi tabel estimasi jumlah bangunan terdampak banjir per desa/kelurahan

flooded_building_per_admin_gdf = admin_gdf.copy()
flooded_building_per_admin_gdf['NUM_BUILDINGS'] = num_buildings
flooded_building_per_admin_gdf = flooded_building_per_admin_gdf[['WADMPR','WADMKK','WADMKC','WADMKD', 'NUM_BUILDINGS']]
flooded_building_per_admin_gdf = flooded_building_per_admin_gdf.rename(columns={'WADMPR': 'Provinsi', 'WADMKK': 'Kabupaten/Kota', 'WADMKC': 'Kecamatan', 'WADMKD': 'Desa/Kelurahan', 'NUM_BUILDINGS': 'Jumlah Bangunan Terdampak'})
flooded_building_per_admin_gdf

In [ ]:
# Jika diperlukan, tabel estimasi jumlah bangunan terdampak banjir per desa/kelurahan dapat diekspor ke file Excel
# File Excel akan tersimpan di dalam Google Drive

output_excel_path = path + 'estimated_flooded_buildings_summary.xlsx'  # Sesuaikan nama filenya
flooded_building_per_admin_gdf.to_excel(output_excel_path, index=False)

In [ ]:
# Kalkulasi total bangunan terdampak banjir untuk seluruh wilayah administrasi
# Peringatan: Proses ini dapat memakan waktu yang sangat lama, bahkan gagal, jika area tergenang banjir sangat luas

total_flooded_buildings_in_region = flooded_buildings.size().getInfo()
print(f"Estimasi Total Jumlah Bangunan Terdampak: {total_flooded_buildings_in_region}")

***Catatan:***

*Estimasi total jumlah bangunan terdampak banjir per desa/kelurahan dapat berbeda dari tabel estimasi jumlah bangunan terdampak banjir per desa/kelurahan sebelumnya jika jumlah bangunan terdampaknya ditotalkan untuk seluruh desa/kelurahan. Hal ini karena jika ada bangunan yang terletak di perbatasan desa/kelurahan (misal bagian depan bangunan masuk Desa A dan bagian belakangnya masuk Desa B), maka bangunan tersebut akan terhitung 2 kali. Jadi jangan menjumlahkan data dari tabel untuk mengetahui total jumlah bangunan terdampak, sebab hasilnya bisa over estimate.*

### **Tahap 4c: Estimasi Dampak Banjir Terhadap Penutupan Lahan**

***Catatan:***

*Estimasi dampak banjir terhadap penutupan lahan hanya dapat dilakukan jika Anda memiliki shapefile penutupan lahan di wilayah yang tergenang banjir, dengan waktu yang sama dengan waktu kejadian banjir. Di sini yang dijadikan sumber data adalah data penutupan lahan dari Kementerian Lingkungan Hidup dan Kehutanan (KLHK) tahun 2021, sesuai dengan waktu kejadian banjir.*

In [ ]:
# Akses shapefile penutupan lahan
# Pastikan shapefile penutupan lahan di simpan di dalam Google Drive

tuplah_gdf = gpd.read_file(path + 'Penutupan_Lahan.shp')      # Sesuaikan nama shapefilenya
tuplah_gdf.head()

In [ ]:
# Visualisasi shapefile penutupan lahan

cover_column = 'Tuplah'     # Sesuaikan nama field/kolom yang memuat data penutupan lahan sebagaimana tabel di atas

tuplah_gdf.plot(column=cover_column, cmap='tab20b', legend=True, figsize=(10, 10))

In [ ]:
# Visualisasi penutupan lahan terdampak banjir

flooded_area_gdf['Remark'] = 'Area Tergenang Banjir'

m = tuplah_gdf.explore(
    column=cover_column,
    cmap='tab20b',
    tooltip=[cover_column],
    popup=True,
    tiles='CartoDB Voyager',
    name='Penutupan Lahan'
)

flooded_area_gdf.explore(
    m=m,
    color='blue',
    style_kwds={'fillOpacity': 0.5, 'weight': 1.5},
    name='Flooded Area Extent',
    tooltip='Remark',
    popup=True
)

m

In [ ]:
# Kalkulasi dan visualisasi luas penutupan lahan terdampak banjir

tuplah_gdf_proj = tuplah_gdf.to_crs('EPSG:' + utm_epsg)
tuplah_flooded_gdf_proj = gpd.overlay(tuplah_gdf_proj, flooded_area_gdf_proj, how='intersection')

tuplah_flooded_gdf_proj['AREA_HA'] = round(tuplah_flooded_gdf_proj.geometry.area / 10000, 2)
tuplah_flooded_gdf_proj = tuplah_flooded_gdf_proj[[cover_column, 'AREA_HA']].rename(columns={cover_column: 'Penutupan Lahan', 'AREA_HA': 'Luas Terdampak (hektare)'})
tuplah_flooded_gdf_proj.sort_values(by='Luas Terdampak (hektare)', ascending=False).reset_index(drop=True)

In [ ]:
# Jika diperlukan, tabel estimasi penutupan lahan terdampak banjir per desa/kelurahan dapat diekspor ke file Excel
# File Excel akan tersimpan di dalam Google Drive

output_excel_path = path + 'estimated_flooded_landcover_summary.xlsx'  # Sesuaikan nama filenya
tuplah_flooded_gdf_proj.to_excel(output_excel_path, index=False)

In [ ]:
# Visualisasi grafik persentase luas penutupan lahan terdampak banjir

import plotly.graph_objects as go

labels = tuplah_flooded_gdf_proj['Penutupan Lahan']
sizes = tuplah_flooded_gdf_proj['Luas Terdampak (hektare)']

fig = go.Figure(data=[go.Pie(labels=labels, values=sizes, pull=[0.05]*len(labels))])
fig.update_layout(height=600, width=800, title_text='Distribusi Luas Lahan Terdampak Banjir Berdasarkan Penutupan Lahan', template='seaborn')
fig.show()